# The Tortilla Index

## What is it and why is it important?
The **Tortilla Index** measures how many kilograms of tortillas can be purchased with exactly one day of the General Minimum Wage (SMG) in Mexico. 

Tortillas are the undisputed foundation of the Mexican diet. For millions of lower-income families, corn tortillas provide the primary source of daily carbohydrates, calories, and calcium. Therefore, the Tortilla Index is not just an economic metric—it is a direct indicator of **national food security** and the real survival purchasing power of the working class.

When this index falls, it means families are forced to work more hours just to secure their most basic caloric needs. When it rises, it signals an alleviation of poverty and an increase in disposable income.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
import pmdarima as pm
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

## 1. Load Data
We strictly filter for **Mom and Pop Stores (Tortillerías)**, because large supermarkets use tortillas as loss-leaders (subsidizing the cost heavily), which distorts the true retail price paid by the majority of the working class.

In [ ]:
df = pd.read_csv('data/tortilla_prices.csv')
df = df.dropna(subset=['Price per kilogram'])

# Filter for Mom and Pop Stores to avoid supermarket distortions
df = df[df['Store type'] == 'Mom and Pop Store']

df['Date'] = pd.to_datetime(df[['Year', 'Month', 'Day']])

# Aggregate average national price by month
monthly_price = df.groupby(['Year', 'Month'])['Price per kilogram'].mean().reset_index()
monthly_price['Date'] = pd.to_datetime(monthly_price[['Year', 'Month']].assign(DAY=1))
monthly_price = monthly_price.set_index('Date').sort_index()

wage_data = {
    'Year': [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
    'SMG_Wage': [50.57, 52.59, 54.80, 57.46, 59.82, 62.33, 64.76, 67.29, 70.10, 73.04, 80.04, 88.36, 102.68, 123.22, 141.70, 172.87, 207.44, 248.93]
}
df_wage = pd.DataFrame(wage_data)

merged = pd.merge(monthly_price.reset_index(), df_wage, on='Year', how='left')
merged = merged.set_index('Date').sort_index()

## 2. Calculate The Index
We calculate the index simply by dividing the daily minimum wage by the average price of one kilogram of tortillas.

In [ ]:
merged['Tortilla Index (kg/day)'] = merged['SMG_Wage'] / merged['Price per kilogram']
ts = merged['Tortilla Index (kg/day)'].dropna()
merged[['Year', 'Month', 'Price per kilogram', 'SMG_Wage', 'Tortilla Index (kg/day)']].tail()

## 3. Visualize the Tortilla Index

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(ts, label='National Tortilla Index (Mom & Pop)', color='goldenrod', linewidth=2)

# Highlight lowest point
min_idx = ts.idxmin()
plt.scatter(min_idx, ts[min_idx], color='red', s=100, zorder=5, label=f"Lowest Purchasing Power ({min_idx.year})")

plt.title('The Tortilla Index (Traditional): Kilograms per Minimum Wage (2007 - 2024)', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Tortilla Index (kg)', fontsize=12)
plt.legend(loc='upper left', fontsize=12)
plt.show()

## 4. Forecasting Purchasing Power with ARIMA
We use the Auto-ARIMA model to forecast how many kilograms a worker will be able to buy over the next 24 months, assuming current trends in wage policy and tortilla price inflation hold.

In [ ]:
adf_result = adfuller(ts)
print(f'ADF Statistic: {adf_result[0]:.4f}')
print(f'p-value: {adf_result[1]:.4f}')
if adf_result[1] > 0.05:
    print("The series is not stationary. Differencing is required.")
else:
    print("The series is stationary.")

In [ ]:
auto_model = pm.auto_arima(ts, start_p=1, start_q=1,
                           max_p=3, max_q=3, m=12,
                           start_P=0, seasonal=True,
                           d=1, D=1, trace=True,
                           error_action='ignore',  
                           suppress_warnings=True, 
                           stepwise=True)
print(auto_model.summary())

In [ ]:
forecast_steps = 24
forecast, conf_int = auto_model.predict(n_periods=forecast_steps, return_conf_int=True)

forecast_index = pd.date_range(start=ts.index[-1] + pd.DateOffset(months=1), periods=forecast_steps, freq='MS')
forecast_series = pd.Series(forecast.values, index=forecast_index)
lower_series = pd.Series(conf_int[:, 0], index=forecast_index)
upper_series = pd.Series(conf_int[:, 1], index=forecast_index)

plt.figure(figsize=(12, 6))
plt.plot(ts, label='Historical Tortilla Index', color='goldenrod', linewidth=2)
plt.plot(forecast_series, label='ARIMA Forecast', color='blue', linewidth=2)
plt.fill_between(forecast_index, lower_series, upper_series, color='blue', alpha=0.2, label='95% Confidence Interval')

plt.title('Tortilla Index (Traditional): Historical & 2-Year Forecast (kg/day)', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Tortilla Index (kg/day)', fontsize=12)
plt.legend(loc='upper left')
plt.show()